# Qwen3.5-2B-Coder — validated Colab v1
Use an A100 40 GB or A100/H100 80 GB runtime. Each expensive step is a separate cell so a failed quality gate cannot accidentally start Stage 2.

In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/coder_SFT')  # Upload or clone this folder here.
WORK_ROOT = Path('/content/drive/MyDrive/qwen35-2b-coder')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
assert (PROJECT_ROOT / 'pyproject.toml').exists(), f'Missing project at {PROJECT_ROOT}'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


AssertionError: Missing project at /content/coder_SFT

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-build-isolation', '-r', str(PROJECT_ROOT / 'requirements-kernels.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--no-deps', '-r', str(PROJECT_ROOT / 'requirements-accelerator.txt')], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-e', str(PROJECT_ROOT)], check=True)
print('Restart the runtime once if Torch or CUDA packages were replaced, then continue below.')

In [ ]:
import os
os.environ['CODER_SFT_WORKDIR'] = str(WORK_ROOT)
for secret in ('HF_TOKEN', 'WANDB_API_KEY'):
    try:
        value = userdata.get(secret)
        if value: os.environ[secret] = value
    except Exception:
        pass
BASE = str(PROJECT_ROOT / 'configs/base.yaml')
DATA = str(PROJECT_ROOT / 'configs/data_v1.yaml')
HARDWARE = str(PROJECT_ROOT / 'configs/hardware.yaml')
STAGE1 = str(PROJECT_ROOT / 'configs/stage1_8k.yaml')
STAGE2 = str(PROJECT_ROOT / 'configs/stage2_repo_32k.yaml')
from coder_sft.config import load_config, write_resolved_config
from coder_sft.hardware import discover_hardware, resolve_hardware
from coder_sft.utils import environment_manifest, write_json
preflight = resolve_hardware(load_config(BASE, HARDWARE, STAGE1), 'stage1', discover_hardware())
(WORK_ROOT / 'reports').mkdir(exist_ok=True)
write_resolved_config(preflight, WORK_ROOT / 'reports/preflight_resolved.yaml')
write_json(environment_manifest(), WORK_ROOT / 'reports/preflight_environment.json')
print(preflight['resolved_hardware'])

## 1. Prepare datasets
Repository reconstruction is network/disk intensive and can be run in a separate CPU session.

In [ ]:
# Optional eight-example, one-step GPU smoke test. Full preparation below replaces the tiny data file.
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run(['prepare-data', '--config', BASE, '--config', DATA, '--limit', '8'], check=True)
    subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'none', '--limit', '8', '--max-steps', '1', '--output-dir', str(WORK_ROOT / 'outputs/smoke')], check=True)

In [ ]:
subprocess.run(['prepare-data', '--config', BASE, '--config', DATA], check=True)
# Uncomment after Stage-1 data succeeds:
# subprocess.run(['build-repo-context', '--config', BASE, '--config', DATA], check=True)

## 2. Baseline generation and Stage 1
Generated benchmark code is saved only; it is never executed in this notebook.

In [ ]:
REPORTS = WORK_ROOT / 'reports'; REPORTS.mkdir(exist_ok=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'humaneval', '--output', str(REPORTS / 'base_humaneval.jsonl')], check=True)
subprocess.run(['generate-eval', '--config', BASE, '--config', DATA, '--model', 'unsloth/Qwen3.5-2B', '--dataset', 'mbpp', '--output', str(REPORTS / 'base_mbpp.jsonl')], check=True)

In [ ]:
subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE1, '--stage', 'stage1', '--resume', 'auto'], check=True)

## 3. Memory profile and gated Stage 2
Score Stage 1 externally and save `reports/stage1_gate.json` with `compare-runs --output`. Stage 2 verifies that this report passed before it starts.

In [ ]:
subprocess.run(['profile-memory', '--config', BASE, '--config', HARDWARE, '--config', STAGE2], check=True)
STAGE1_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s1-8k/final_adapter'
# After writing base.json and stage1.json from external EvalPlus scores:
# subprocess.run(['compare-runs', '--baseline', str(REPORTS / 'base.json'), '--candidate', str(REPORTS / 'stage1.json'), '--stage', 'stage1', '--output', str(REPORTS / 'stage1_gate.json')], check=True)
# After that gate passes and repo_v1.jsonl exists:
# subprocess.run(['train-sft', '--config', BASE, '--config', HARDWARE, '--config', STAGE2, '--stage', 'stage2', '--adapter', str(STAGE1_ADAPTER), '--resume', 'auto'], check=True)

## 4. Export
Merged output is verified against the adapter with deterministic prompts before GGUF export is allowed.

In [ ]:
STAGE2_ADAPTER = WORK_ROOT / 'outputs/q35-2b-coder-s2-repo/final_adapter'
# subprocess.run(['export-model', '--adapter', str(STAGE2_ADAPTER), '--output', str(WORK_ROOT / 'exports/q35-coder-merged'), '--format', 'merged'], check=True)